## 🔧 Step 1: Setup and Mount Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify GPU availability
import torch
print(f"🔥 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🚀 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU not available - please enable GPU runtime")

## 📦 Step 2: Install Required Libraries

In [ ]:
# Install required packages
!pip install torch torchvision transformers
!pip install opencv-python matplotlib seaborn
!pip install scikit-learn pillow numpy pandas
!pip install pytorch-grad-cam
!pip install tqdm

print("✅ All packages installed successfully!")

## 📂 Step 3: Setup Paths and Configuration

In [ ]:
# Import libraries
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from PIL import Image
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time
import json
import random

# Deep Learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from transformers import ViTForImageClassification, ViTImageProcessor
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast

# Evaluation
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

print(f"📦 PyTorch version: {torch.__version__}")
print(f"🔥 CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Configuration for Google Drive paths
DRIVE_PATH = "/content/drive/My Drive"
DATASET_PATH = f"{DRIVE_PATH}/data"  # Your dataset folder in Drive
MODEL_SAVE_PATH = f"{DRIVE_PATH}/models"  # Where to save trained models

# Create model directory if it doesn't exist
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

# Defect classes (Matching the folder names in the new dataset)
CLASS_NAMES = ['Crazing', 'Inclusion', 'Patches', 'Pitted', 'Rolled', 'Scratches']
NUM_CLASSES = len(CLASS_NAMES)

# Training configuration
IMAGE_SIZE = 224
BATCH_SIZE = 16  # Optimized for GPU
EPOCHS = 10  # Reduced for faster training
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

print("🔧 Configuration:")
print(f"   📁 Dataset Path: {DATASET_PATH}")
print(f"   💾 Model Save Path: {MODEL_SAVE_PATH}")
print(f"   🏷️ Classes: {CLASS_NAMES}")
print(f"   📐 Image Size: {IMAGE_SIZE}×{IMAGE_SIZE}")
print(f"   📦 Batch Size: {BATCH_SIZE}")
print(f"   🔄 Epochs: {EPOCHS}")

## 📊 Step 4: Verify Dataset and Create Data Loaders

In [ ]:
# Verify dataset structure
def verify_dataset():
    """Verify the dataset structure in Google Drive"""
    print("🔍 Verifying dataset structure...")

    if not os.path.exists(DATASET_PATH):
        print(f"❌ Dataset path not found: {DATASET_PATH}")
        print("📝 Please ensure your dataset is uploaded to Google Drive in the 'data' folder")
        return False

    # Check for split folders
    splits = ['train', 'valid', 'test']
    total_images = 0

    for split in splits:
        split_path = os.path.join(DATASET_PATH, split)
        if not os.path.exists(split_path):
            print(f"❌ Split folder not found: {split}")
            continue

        print(f"   📂 Checking {split} split:")
        for class_name in CLASS_NAMES:
            class_path = os.path.join(split_path, class_name)
            if os.path.exists(class_path):
                image_files = [f for f in os.listdir(class_path) if f.endswith(('.png', '.jpg', '.jpeg'))]
                total_images += len(image_files)
                print(f"      ✅ {class_name}: {len(image_files)} images")
            else:
                print(f"      ❌ {class_name}: folder not found")

    print(f"\n📊 Total images found: {total_images}")
    return total_images > 0

# Verify dataset
dataset_ok = verify_dataset()

In [ ]:
print(f"Listing contents of: {DATASET_PATH}")
!ls -R "{DATASET_PATH}"

In [ ]:
# Verify dataset structure
def verify_dataset():
    """Verify the dataset structure in Google Drive"""
    print("🔍 Verifying dataset structure...")

    if not os.path.exists(DATASET_PATH):
        print(f"❌ Dataset path not found: {DATASET_PATH}")
        print("📝 Please ensure your dataset is uploaded to Google Drive in the 'data' folder")
        return False

    # Check for split folders
    splits = ['train', 'valid', 'test']
    total_images = 0

    for split in splits:
        split_path = os.path.join(DATASET_PATH, split)
        if not os.path.exists(split_path):
            print(f"❌ Split folder not found: {split}")
            continue

        print(f"   📂 Checking {split} split:")
        for class_name in CLASS_NAMES:
            class_path = os.path.join(split_path, class_name)
            if os.path.exists(class_path):
                image_files = [f for f in os.listdir(class_path) if f.endswith(('.png', '.jpg', '.jpeg', '.bmp'))]
                total_images += len(image_files)
                print(f"      ✅ {class_name}: {len(image_files)} images")
            else:
                print(f"      ❌ {class_name}: folder not found")

    print(f"\n📊 Total images found: {total_images}")
    return total_images > 0

# Verify dataset
dataset_ok = verify_dataset()

In [ ]:
# Custom Dataset Class
class DefectDataset(Dataset):
    """Custom PyTorch Dataset for NEU Surface Defect Classification"""

    def __init__(self, data_path, class_names, transform=None, mode='train'):
        self.data_path = data_path
        self.class_names = class_names
        self.transform = transform
        self.mode = mode

        # Create class to index mapping
        self.class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}

        # Collect all image paths and labels
        self.image_paths = []
        self.labels = []

        for class_name in class_names:
            class_path = os.path.join(data_path, class_name)
            if os.path.exists(class_path):
                for filename in os.listdir(class_path):
                    if filename.endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                        self.image_paths.append(os.path.join(class_path, filename))
                        self.labels.append(self.class_to_idx[class_name])

        print(f"📊 {mode.capitalize()} Dataset: {len(self.image_paths)} images loaded")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load image
        img_path = self.image_paths[idx]
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        # Convert to RGB (3 channels) for ViT
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        image = Image.fromarray(image)

        # Apply transforms
        if self.transform:
            image = self.transform(image)

        label = self.labels[idx]
        return image, label

# Data transforms
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("✅ Dataset class and transforms defined")

In [ ]:
# Create train/validation split and data loaders
def create_data_loaders():
    """Create training and validation data loaders"""

    if not dataset_ok:
        print("❌ Cannot create data loaders - dataset not found")
        return None, None

    # Define paths for train and validation sets
    train_path = os.path.join(DATASET_PATH, 'train')
    val_path = os.path.join(DATASET_PATH, 'valid')

    # Create datasets
    train_dataset = DefectDataset(train_path, CLASS_NAMES, transform=train_transforms, mode='train')
    val_dataset = DefectDataset(val_path, CLASS_NAMES, transform=val_transforms, mode='validation')

    # Create data loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    print(f"📊 Data Loaders Created:")
    print(f"   🎯 Training samples: {len(train_dataset)}")
    print(f"   🔍 Validation samples: {len(val_dataset)}")
    print(f"   📦 Batch size: {BATCH_SIZE}")

    return train_loader, val_loader

# Create data loaders
train_loader, val_loader = create_data_loaders()

In [ ]:
# Vision Transformer Model
class ExplainableViTDefectClassifier(nn.Module):
    """Vision Transformer for Defect Detection"""

    def __init__(self, num_classes=6, model_name="google/vit-base-patch16-224-in21k"):
        super().__init__()

        # Load pre-trained ViT model
        self.vit = ViTForImageClassification.from_pretrained(
            model_name,
            num_labels=num_classes,
            ignore_mismatched_sizes=True
        )

        print(f"🤖 Loaded ViT model: {model_name}")
        print(f"🎯 Number of classes: {num_classes}")

    def forward(self, x):
        outputs = self.vit(x)
        return outputs

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ExplainableViTDefectClassifier(num_classes=NUM_CLASSES)
model = model.to(device)

print(f"🔥 Model initialized on: {device}")

# Model parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"📊 Model Parameters:")
print(f"   📈 Total parameters: {total_params:,}")
print(f"   🎯 Trainable parameters: {trainable_params:,}")

## 🤖 Step 5: Initialize Vision Transformer Model

In [ ]:
# Vision Transformer Model
class ExplainableViTDefectClassifier(nn.Module):
    """Vision Transformer for Defect Detection"""

    def __init__(self, num_classes=6, model_name="google/vit-base-patch16-224-in21k"):
        super().__init__()

        # Load pre-trained ViT model
        self.vit = ViTForImageClassification.from_pretrained(
            model_name,
            num_labels=num_classes,
            ignore_mismatched_sizes=True
        )

        print(f"🤖 Loaded ViT model: {model_name}")
        print(f"🎯 Number of classes: {num_classes}")

    def forward(self, x):
        outputs = self.vit(x)
        return outputs

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ExplainableViTDefectClassifier(num_classes=NUM_CLASSES)
model = model.to(device)

print(f"🔥 Model initialized on: {device}")

# Model parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"📊 Model Parameters:")
print(f"   📈 Total parameters: {total_params:,}")
print(f"   🎯 Trainable parameters: {trainable_params:,}")

## 🏋️ Step 6: Training Loop with Mixed Precision

In [ ]:
# Setup training components
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# Mixed precision training
scaler = GradScaler()

print("🔧 Training components initialized:")
print(f"   📉 Loss function: CrossEntropyLoss")
print(f"   🚀 Optimizer: AdamW (lr={LEARNING_RATE})")
print(f"   📊 Scheduler: CosineAnnealingLR")
print(f"   ⚡ Mixed precision: Enabled")

In [ ]:
# Training functions
def train_epoch(model, train_loader, criterion, optimizer, scaler, device, epoch):
    """Train the model for one epoch with mixed precision"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for batch_idx, (data, target) in enumerate(progress_bar):
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()

        # Mixed precision forward pass
        with autocast():
            outputs = model(data)
            loss = criterion(outputs.logits, target)

        # Mixed precision backward pass
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        pred = outputs.logits.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += target.size(0)

        # Update progress bar
        accuracy = 100. * correct / total
        avg_loss = total_loss / (batch_idx + 1)
        gpu_memory = f'{torch.cuda.memory_allocated()/1e9:.1f}GB' if torch.cuda.is_available() else 'CPU'
        progress_bar.set_postfix({
            'Loss': f'{avg_loss:.4f}',
            'Acc': f'{accuracy:.2f}%',
            'GPU': gpu_memory
        })

    return total_loss / len(train_loader), accuracy

def validate_epoch(model, val_loader, criterion, device):
    """Validate the model"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    class_correct = [0] * NUM_CLASSES
    class_total = [0] * NUM_CLASSES

    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            outputs = model(data)
            loss = criterion(outputs.logits, target)

            total_loss += loss.item()
            pred = outputs.logits.argmax(dim=1)
            correct += pred.eq(target).sum().item()
            total += target.size(0)

            # Per-class accuracy
            for i in range(target.size(0)):
                label = target[i]
                class_correct[label] += pred[i].eq(target[i]).item()
                class_total[label] += 1

    accuracy = 100. * correct / total

    # Print per-class accuracy
    print("\n📊 Per-class Validation Accuracy:")
    for i, class_name in enumerate(CLASS_NAMES):
        if class_total[i] > 0:
            class_acc = 100. * class_correct[i] / class_total[i]
            print(f"   {class_name.replace('_', ' ').title()}: {class_acc:.1f}%")

    return total_loss / len(val_loader), accuracy

print("✅ Training functions defined")

In [ ]:
# Main training loop
if train_loader is not None and val_loader is not None:
    print("🚀 Starting model training...")
    print(f"⏱️ Estimated training time: {EPOCHS * 3:.0f}-{EPOCHS * 5:.0f} minutes")

    best_val_accuracy = 0
    best_model_state = None
    training_start = time.time()

    # Training metrics
    train_losses = []
    train_accuracies = []
    val_losses = []
    val_accuracies = []

    for epoch in range(EPOCHS):
        epoch_start = time.time()

        print(f"\n📊 Epoch {epoch+1}/{EPOCHS}")
        print("-" * 50)

        # Training phase
        train_loss, train_acc = train_epoch(
            model, train_loader, criterion, optimizer, scaler, device, epoch
        )

        # Validation phase
        val_loss, val_acc = validate_epoch(model, val_loader, criterion, device)

        # Learning rate scheduling
        scheduler.step()

        # Track metrics
        train_losses.append(train_loss)
        train_accuracies.append(train_acc)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)

        # Save best model to Google Drive
        if val_acc > best_val_accuracy:
            best_val_accuracy = val_acc
            best_model_state = model.state_dict().copy()
            torch.save(best_model_state, f'{MODEL_SAVE_PATH}/best_vit_defect_detector.pt')
            print(f"💾 New best model saved to Drive! Validation accuracy: {val_acc:.2f}%")

        # Epoch summary
        epoch_time = time.time() - epoch_start
        total_elapsed = time.time() - training_start
        remaining_time = ((total_elapsed / (epoch + 1)) * (EPOCHS - epoch - 1)) / 60

        print(f"📈 Training   - Loss: {train_loss:.4f}, Accuracy: {train_acc:.2f}%")
        print(f"📊 Validation - Loss: {val_loss:.4f}, Accuracy: {val_acc:.2f}%")
        print(f"⏱️ Epoch time: {epoch_time/60:.1f}min | Remaining: {remaining_time:.1f}min")
        print(f"🎯 Current LR: {optimizer.param_groups[0]['lr']:.6f}")

        # Save checkpoint every 3 epochs
        if (epoch + 1) % 3 == 0:
            checkpoint_path = f'{MODEL_SAVE_PATH}/checkpoint_epoch_{epoch+1}.pt'
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': train_loss,
                'val_loss': val_loss,
                'val_accuracy': val_acc
            }, checkpoint_path)
            print(f"💾 Checkpoint saved: epoch_{epoch+1}.pt")

    print(f"\n🎉 Training completed!")
    print(f"🏆 Best validation accuracy: {best_val_accuracy:.2f}%")
    print(f"⏱️ Total training time: {(time.time() - training_start)/60:.1f} minutes")

    # Load best model
    if best_model_state:
        model.load_state_dict(best_model_state)
        print(f"✅ Best model loaded for final evaluation")

    # Save training metrics
    training_history = {
        'train_losses': train_losses,
        'train_accuracies': train_accuracies,
        'val_losses': val_losses,
        'val_accuracies': val_accuracies,
        'best_val_accuracy': best_val_accuracy,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE
    }

    with open(f'{MODEL_SAVE_PATH}/training_history.json', 'w') as f:
        json.dump(training_history, f, indent=2)
    print(f"📊 Training history saved to Drive")

else:
    print("❌ Cannot start training - data loaders not created")

## 📊 Step 7: Final Evaluation and Visualization

In [ ]:
# Final evaluation
if train_loader is not None and val_loader is not None:
    print("📊 Final Model Evaluation")
    print("=" * 40)

    # Comprehensive evaluation
    model.eval()
    all_predictions = []
    all_labels = []
    all_probabilities = []

    with torch.no_grad():
        for data, target in tqdm(val_loader, desc="Final Evaluation"):
            data, target = data.to(device), target.to(device)
            outputs = model(data)

            probabilities = F.softmax(outputs.logits, dim=1)
            predictions = outputs.logits.argmax(dim=1)

            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(target.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())

    # Calculate metrics
    y_true = np.array(all_labels)
    y_pred = np.array(all_predictions)

    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, average=None)

    print(f"\n🎯 Final Results:")
    print(f"   📈 Overall Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"   📊 Average F1-Score: {np.mean(f1):.4f}")

    # Per-class results
    print(f"\n📋 Per-Class Results:")
    for i, class_name in enumerate(CLASS_NAMES):
        print(f"   {class_name.replace('_', ' ').title()}:")
        print(f"     Precision: {precision[i]:.3f}")
        print(f"     Recall: {recall[i]:.3f}")
        print(f"     F1-Score: {f1[i]:.3f}")
        print(f"     Support: {support[i]}")

    # Save evaluation results
    evaluation_results = {
        'accuracy': float(accuracy),
        'precision': precision.tolist(),
        'recall': recall.tolist(),
        'f1_score': f1.tolist(),
        'support': support.tolist(),
        'class_names': CLASS_NAMES
    }

    with open(f'{MODEL_SAVE_PATH}/evaluation_results.json', 'w') as f:
        json.dump(evaluation_results, f, indent=2)

    print(f"\n💾 Evaluation results saved to Drive")
    print(f"📁 Model files saved to: {MODEL_SAVE_PATH}")

In [ ]:
# Plot training progress
if 'train_losses' in locals():
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Loss plot
    axes[0].plot(range(1, len(train_losses)+1), train_losses, 'b-', label='Training Loss', linewidth=2)
    axes[0].plot(range(1, len(val_losses)+1), val_losses, 'r-', label='Validation Loss', linewidth=2)
    axes[0].set_title('📉 Training and Validation Loss', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Accuracy plot
    axes[1].plot(range(1, len(train_accuracies)+1), train_accuracies, 'b-', label='Training Accuracy', linewidth=2)
    axes[1].plot(range(1, len(val_accuracies)+1), val_accuracies, 'r-', label='Validation Accuracy', linewidth=2)
    axes[1].set_title('📈 Training and Validation Accuracy', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{MODEL_SAVE_PATH}/training_progress.png', dpi=300, bbox_inches='tight')
    plt.show()

    print("📊 Training progress visualization saved to Drive")

## ✅ Training Complete!

### 🎉 **Success!** Your Vision Transformer model has been trained and saved to Google Drive.

### 📁 **Files Saved to Your Drive:**
- `best_vit_defect_detector.pt` - Best model weights
- `checkpoint_epoch_*.pt` - Training checkpoints  
- `training_history.json` - Training metrics
- `evaluation_results.json` - Final evaluation results
- `training_progress.png` - Training visualization

### 🔄 **Next Steps:**
1. **Download the trained model** from your Google Drive
2. **Use it in your local notebook** for explainability features
3. **Deploy with Streamlit** for interactive defect detection

### 📊 **Model Performance:**
The model should achieve **85-95% accuracy** on the NEU Surface Defect Dataset, ready for production use with explainable AI features!